## Qwen Quick Start Notebook

This notebook shows how to train and infer the Qwen-7B-Chat model on a single GPU. Similarly, Qwen-1.8B-Chat, Qwen-14B-Chat can also be leveraged for the following steps. We only need to modify the corresponding `model name` and hyper-parameters. The training and inference of Qwen-72B-Chat requires higher GPU requirements and larger disk space.

## Requirements
- Python 3.8 and above
- Pytorch 1.12 and above, 2.0 and above are recommended
- CUDA 11.4 and above are recommended (this is for GPU users, flash-attention users, etc.)
We test the training of the model on an A10 GPU (24GB).

## Extra
If you need to speed up, you can install  `flash-attention`. The details of the installation can be found [here](https://github.com/Dao-AILab/flash-attention).

In [ ]:
!git clone https://github.com/Dao-AILab/flash-attention
!cd flash-attention && pip install .
# Below are optional. Installing them might be slow.
# !pip install csrc/layer_norm
# If the version of flash-attn is higher than 2.1.1, the following is not needed.
# !pip install csrc/rotary

### Step 0: Install Package Requirements

In [ ]:
!pip install transformers>=4.32.0 accelerate tiktoken einops scipy transformers_stream_generator==0.0.4 peft deepspeed modelscope

### Step 1: Download Model
When using `transformers` in some regions, the model cannot be automatically downloaded due to network problems. We recommend using `modelscope` to download the model first, and then use `transformers` for inference.

In [ ]:
from modelscope import snapshot_download

# Downloading model checkpoint to a local dir model_dir.
model_dir = snapshot_download('Qwen/Qwen-7B-Chat', cache_dir='.', revision='master')

### Step 2: Direct Model Inference 
We recommend two ways to do model inference: `modelscope` and `transformers`.

#### 2.1 Model Inference with ModelScope

In [ ]:
from modelscope import AutoModelForCausalLM, AutoTokenizer
from modelscope import GenerationConfig

# Note: The default behavior now has injection attack prevention off.
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen-7B-Chat/", trust_remote_code=True)

# use bf16
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat/", device_map="auto", trust_remote_code=True, bf16=True).eval()
# use fp16
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat/", device_map="auto", trust_remote_code=True, fp16=True).eval()
# use cpu only
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat/", device_map="cpu", trust_remote_code=True).eval()
# use auto mode, automatically select precision based on the device.
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat/", device_map="auto", trust_remote_code=True).eval()

# Specify hyperparameters for generation
model.generation_config = GenerationConfig.from_pretrained("Qwen/Qwen-7B-Chat/", trust_remote_code=True)

# First round conversation.
# Q: 你好
# A: 你好！很高兴见到你。
response, history = model.chat(tokenizer, "你好", history=None)
print(response)
# Second round conversation.
response, history = model.chat(tokenizer, "给我讲一个年轻人奋斗创业最终取得成功的故事", history=history)
print(response)
# Third round conversation.
response, history = model.chat(tokenizer, "给这个故事起一个标题", history=history)
print(response)

#### 2.2 Model Inference with Transformers

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.generation import GenerationConfig

# Note: The default behavior now has injection attack prevention off.
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen-7B-Chat/", trust_remote_code=True)

# use bf16
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat/", device_map="auto", trust_remote_code=True, bf16=True).eval()
# use fp16
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat/", device_map="auto", trust_remote_code=True, fp16=True).eval()
# use cpu only
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat/", device_map="cpu", trust_remote_code=True).eval()
# use auto mode, automatically select precision based on the device.
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat/", device_map="auto", trust_remote_code=True).eval()

# Specify hyperparameters for generation
model.generation_config = GenerationConfig.from_pretrained("Qwen/Qwen-7B-Chat/", trust_remote_code=True)

# First round conversation
response, history = model.chat(tokenizer, "你好", history=None)
print(response)
# Second round conversation.
response, history = model.chat(tokenizer, "给我讲一个年轻人奋斗创业最终取得成功的故事", history=history)
print(response)
# Third round conversation.
response, history = model.chat(tokenizer, "给这个故事起一个标题", history=history)
print(response)

### Step 3: Fine-Tune with QLoRA

We support using QLoRA to fine-tune the Qwen model. The process is as follows:

#### 3.1 Install Requirements

In [ ]:
!pip install peft

#### 3.2 Prepare Data

We use a self-instruct dataset as an example, where each sample contains an instruction and a response. The data is in the following format:

In [ ]:
train_data = {
    "instruction": "翻译以下句子：Hello, how are you?",
    "input": "",
    "output": "你好，你好吗？"
}

#### 3.3 Train with QLoRA

In [ ]:
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained(
    "Qwen/Qwen-7B-Chat/",
    device_map="auto",
    trust_remote_code=True
).eval()

merged_model = model.merge_and_unload()
merged_model.save_pretrained("output_qwen_merged", max_shard_size="2048MB", safe_serialization=True)

The tokenizer files are not saved in the new directory in this step. You can copy the tokenizer files or use the following code:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen-7B-Chat/",
    trust_remote_code=True
)

tokenizer.save_pretrained("output_qwen_merged")

### 3.4 Test the Model

After merging the weights, we can test the model as follows:

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.generation import GenerationConfig

tokenizer = AutoTokenizer.from_pretrained("output_qwen_merged", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    "output_qwen_merged",
    device_map="auto",
    trust_remote_code=True
).eval()

response, history = model.chat(tokenizer, "你好", history=None)
print(response)